# Data Cleaning -- BRAF+ IO/IO vs. Targeted Therapy
**This notebook prepares Flatiron Health CSV files for patients with BRAF-positive advanced melanoma treated with first-line combination immunotherapy or targeted therapy. Each CSV is cleaned using the flatiron_cleaner package. The cleaned dataframes are then merged into a single dataset, which will be used for survival analysis.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorMelanoma, merge_dataframes

## Import data

In [2]:
df = pd.read_csv('../outputs/ioio_tki_index.csv')

In [3]:
df.head(5)

,PatientID,LineName,StartDate
0,F744F618949B5,ioio,2020-04-24
1,F5C59C9946310,ioio,2018-07-17
2,F1A32DD843C2F,ioio,2020-04-15
3,F75D9B0C72385,ioio,2017-07-13
4,FD927F4EDE1D3,ioio,2022-04-08


In [4]:
df.shape

(2771, 3)

## Clean CSV files 

In [5]:
# Initialize class 
processor = DataProcessorMelanoma()

### Process Enhanced_AdvancedMelanoma.csv

In [6]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvancedMelanoma.csv',
                                         index_date_df = df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2025-11-27 00:11:54,886 - INFO - Successfully read Enhanced_AdvancedMelanoma.csv file with shape: (16997, 18) and unique PatientIDs: 16997
2025-11-27 00:11:54,891 - INFO - Successfully filtered Enhanced_AdvancedMelanoma.csv file with shape: (2771, 19) and unique PatientIDs: 2771
2025-11-27 00:11:54,904 - INFO - Successfully processed Enhanced_AdvancedMelanoma.csv file with final shape: (2771, 21) and unique PatientIDs: 2771


In [7]:
enhanced_df['days_adv_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])
enhanced_df['days_to_treatment_before_30d'] = np.where(enhanced_df['days_adv_to_treatment'] < 30, 1, 0)

In [8]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [9]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
IV         1073
unknown     495
III         458
II          438
I           307
Name: count, dtype: int64

In [10]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [11]:
enhanced_df['adv_diagnosis_year'] = pd.to_numeric(enhanced_df['adv_diagnosis_year'])
enhanced_df['before_2020'] = np.where(enhanced_df['adv_diagnosis_year'] < 2020, 1, 0)

In [12]:
# drop dates 
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'AdvancedDiagnosisDate', 
                                          'MetDiagnosisDate', 
                                          'FirstLocalRecurDate',
                                          'FirstDistantRecurDate', 
                                          'FirstVisceralMetDate',
                                          'FirstNonVisceralMetDate',
                                          'imported_StartDate'])

### Process Demographics.csv 

In [13]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = df,
                                                 index_date_column = 'StartDate')

2025-11-27 00:11:54,939 - INFO - Successfully read Demographics.csv file with shape: (16997, 6) and unique PatientIDs: 16997
2025-11-27 00:11:54,946 - WARNING - Found 3 ages outside valid range (18-120)
2025-11-27 00:11:54,950 - INFO - Successfully processed Demographics.csv file with final shape: (2771, 6) and unique PatientIDs: 2771


In [14]:
demographics_df = demographics_df.query('age >= 18', engine = 'python')

In [15]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M    1872
F     896
Name: count, dtype: int64

In [16]:
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

/var/folders/lr/vkkcj_s12115sxc05ly3mshh0000gn/T/ipykernel_31656/3338690197.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)


In [17]:
demographics_df = demographics_df.drop(columns = 'Gender')

### Process Enhanced_AdvMelanomaBiomarkers.csv

In [18]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvMelanomaBiomarkers.csv',
                                             index_date_df = df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2025-11-27 00:11:55,007 - INFO - Successfully read Enhanced_AdvMelanomaBiomarkers.csv file with shape: (31257, 18) and unique PatientIDs: 13249
2025-11-27 00:11:55,021 - INFO - Successfully merged Enhanced_AdvMelanomaBiomarkers.csv df with index_date_df resulting in shape: (7099, 19) and unique PatientIDs: 2617
2025-11-27 00:11:55,086 - INFO - Successfully processed Enhanced_AdvMelanomaBiomarkers.csv file with final shape: (2771, 6) and unique PatientIDs: 2771


In [19]:
biomarkers_df.BRAF_status.value_counts(dropna = False)

BRAF_status
positive    1343
negative     900
NaN          493
unknown       35
Name: count, dtype: int64

### Process ECOG.csv

In [20]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2025-11-27 00:11:55,178 - INFO - Successfully read ECOG.csv file with shape: (211808, 4) and unique PatientIDs: 12139
2025-11-27 00:11:55,218 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (47325, 5) and unique PatientIDs: 2253
2025-11-27 00:11:55,256 - INFO - Successfully processed ECOG.csv file with final shape: (2771, 3) and unique PatientIDs: 2771


In [21]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
0      915
NaN    907
1      701
2      179
3       65
4        4
Name: count, dtype: int64

In [22]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [23]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [24]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2025-11-27 00:11:59,229 - INFO - Successfully read Vitals.csv file with shape: (4009376, 16) and unique PatientIDs: 16978
2025-11-27 00:12:00,752 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (791719, 17) and unique PatientIDs: 2770
2025-11-27 00:12:01,139 - INFO - Successfully processed Vitals.csv file with final shape: (2771, 8) and unique PatientIDs: 2771


### Process Lab.csv

In [25]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2025-11-27 00:12:15,512 - INFO - Successfully read Lab.csv file with shape: (10819406, 17) and unique PatientIDs: 16150
2025-11-27 00:12:18,145 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (2270176, 18) and unique PatientIDs: 2728
2025-11-27 00:12:23,121 - INFO - Successfully processed Lab.csv file with final shape: (2771, 81) and unique PatientIDs: 2771


### Process MedicationAdministration.csv

In [26]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2025-11-27 00:12:24,004 - INFO - Successfully read MedicationAdministration.csv file with shape: (712643, 11) and unique PatientIDs: 12796
2025-11-27 00:12:24,202 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (154723, 12) and unique PatientIDs: 2482
2025-11-27 00:12:24,235 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (2771, 9) and unique PatientIDs: 2771


### Process Diagnosis.csv

In [27]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2025-11-27 00:12:24,982 - INFO - Successfully read Diagnosis.csv file with shape: (1130923, 6) and unique PatientIDs: 16997
2025-11-27 00:12:25,122 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (173472, 7) and unique PatientIDs: 2771
2025-11-27 00:12:25,411 - INFO - Successfully processed Diagnosis.csv file with final shape: (2771, 31) and unique PatientIDs: 2771


### Process Enhanced_Mortality_V2.csv

In [28]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvMelanomaBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvMelanoma_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvMelanoma_Progression.csv',
                                           metastatic_sites_path = '../data/Enhanced_AdvMelanoma_SitesOfMet.csv',
                                           drop_dates = False)

2025-11-27 00:12:25,426 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (7284, 2) and unique PatientIDs: 7284
2025-11-27 00:12:25,432 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (2771, 3) and unique PatientIDs: 2771
2025-11-27 00:12:25,910 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date', 'last_met_date'] are used to calculate the last EHR date
2025-11-27 00:12:25,914 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (2771, 6) and unique PatientIDs: 2771. There are 0 out of 2771 patients with missing duration values


In [29]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [30]:
mortality_df = mortality_df.query('duration >= 0')

### Process Insurance.csv

In [31]:
insurance_df = processor.process_insurance(file_path = '../data/Insurance.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0,
                                           missing_date_strategy = 'liberal')

2025-11-27 00:12:26,049 - INFO - Successfully read Insurance.csv file with shape: (88152, 14) and unique PatientIDs: 16356
2025-11-27 00:12:26,095 - INFO - Successfully merged Insurance.csv df with index_date_df resulting in shape: (13593, 15) and unique PatientIDs: 2702
2025-11-27 00:12:26,131 - INFO - Successfully processed Insurance.csv file with final shape: (2771, 5) and unique PatientIDs: 2771


### Process SocialDeterminantsOfHealth.csv 

In [32]:
ses_df = pd.read_csv('../data/SocialDeterminantsOfHealth.csv')

In [33]:
ses_df.head(3)

,PatientID,SESIndex2015_2019
0,FD5ED0FAD6B38,2
1,FFF3EA15CF349,2
2,FA7CF55D364A5,3


In [34]:
ses_df.SESIndex2015_2019.value_counts(dropna = False)

SESIndex2015_2019
4                  3827
5 - Highest SES    3759
3                  3348
2                  2768
1 - Lowest SES     1712
NaN                1583
Name: count, dtype: int64

In [35]:
ses_df['ses_mod'] = ses_df['SESIndex2015_2019'].map({
    '1 - Lowest SES' : 1, 
    '2': 2, 
    '3': 3, 
    '4': 4, 
    '5 - Highest SES': 5
})

ses_df['ses_mod_na'] = np.where(ses_df['ses_mod'].isna(), 1, 0)

# impute 3 for missing SES
ses_df['ses_mod'] = ses_df['ses_mod'].fillna(3)

In [36]:
ses_df = ses_df[['PatientID', 'ses_mod', 'ses_mod_na']]

In [37]:
ses_df = ses_df[ses_df.PatientID.isin(df.PatientID)]

### Process Enhanced_AdvMelanomaProcedures.csv

In [38]:
procedures_df = processor.process_procedures(file_path = '../data/Enhanced_AdvMelanomaProcedures.csv',
                                             index_date_df = df,
                                             index_date_column = 'StartDate',
                                             days_before = None,
                                             days_after = 0)

2025-11-27 00:12:26,308 - INFO - Successfully read Enhanced_AdvMelanomaProcedures.csv file with shape: (44284, 3) and unique PatientIDs: 13432
2025-11-27 00:12:26,321 - INFO - Successfully merged Enhanced_AdvMelanomaProcedures.csv df with index_date_df resulting in shape: (4829, 4) and unique PatientIDs: 1681
2025-11-27 00:12:26,329 - INFO - Successfully processed Enhanced_AdvMelanomaProcedures.csv file with final shape: (2771, 4) and unique PatientIDs: 2771


### Process Enhanced_AdvMelanoma_SitesOfMet.csv

In [39]:
metastasis_df = processor.process_metastasis(file_path = '../data/Enhanced_AdvMelanoma_SitesOfMet.csv',
                                             index_date_df = df,
                                             index_date_column = 'StartDate',
                                             days_before = None,
                                             days_after = 0)

2025-11-27 00:12:26,343 - INFO - Successfully read Enhanced_AdvMelanoma_SitesOfMet.csv file with shape: (29302, 3) and unique PatientIDs: 10076
2025-11-27 00:12:26,350 - INFO - Successfully merged Enhanced_AdvMelanoma_SitesOfMet.csv df with index_date_df resulting in shape: (8467, 4) and unique PatientIDs: 2550
2025-11-27 00:12:26,360 - INFO - Successfully processed Enhanced_AdvMelanoma_SitesOfMet.csv file with final shape: (2771, 9) and unique PatientIDs: 2771


## Merge dataframes

In [40]:
ioio_tki_features_df = merge_dataframes(enhanced_df,
                                        demographics_df,
                                        biomarkers_df,
                                        ecog_df,
                                        vitals_df,
                                        labs_df,
                                        medications_df,
                                        diagnosis_df, 
                                        mortality_df, 
                                        insurance_df,
                                        ses_df,
                                        procedures_df,
                                        metastasis_df,
                                        merge_type = 'inner')

2025-11-27 00:12:26,365 - INFO - Anticipated number of merges: 12
2025-11-27 00:12:26,366 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 174
2025-11-27 00:12:26,367 - INFO - Dataset 1 shape: (2771, 17), unique PatientIDs: 2771
2025-11-27 00:12:26,368 - INFO - Dataset 2 shape: (2768, 6), unique PatientIDs: 2768
2025-11-27 00:12:26,368 - INFO - Dataset 3 shape: (2771, 6), unique PatientIDs: 2771
2025-11-27 00:12:26,369 - INFO - Dataset 4 shape: (2771, 4), unique PatientIDs: 2771
2025-11-27 00:12:26,370 - INFO - Dataset 5 shape: (2771, 8), unique PatientIDs: 2771
2025-11-27 00:12:26,370 - INFO - Dataset 6 shape: (2771, 81), unique PatientIDs: 2771
2025-11-27 00:12:26,371 - INFO - Dataset 7 shape: (2771, 9), unique PatientIDs: 2771
2025-11-27 00:12:26,371 - INFO - Dataset 8 shape: (2771, 31), unique PatientIDs: 2771
2025-11-27 00:12:26,372 - INFO - Dataset 9 shape: (2763, 3), unique PatientIDs: 2763
2025-11-27 00:12:26,372 -

In [41]:
ioio_tki_features_df.shape

(2760, 174)

In [42]:
ioio_tki_features_df.head(2)

,PatientID,CalcResectInitialDx,TStage_mod,NStage_mod,MStage_mod,GroupStage_mod,ResidualDiseaseInitialDx_mod,ResidualDiseaseLocalRecur_mod,CalcResectLocalRecur_mod,days_diagnosis_to_adv,...,primary_site_procedure,lymph_node_procedure,other_viscera_met,thoracic_met,other_met,lymph_met,skin_met,bone_met,brain_met,liver_met
0,F744F618949B5,NaN,NaN,NaN,NaN,4.0,NaN,unknown,Unknown,0.0,...,0,0,1,1,1,0,1,0,0,0
1,F4AAE7EB8AE49,NaN,NaN,NaN,NaN,4.0,NaN,unknown,Unknown,0.0,...,0,0,0,0,0,0,0,1,0,1


In [43]:
ioio_tki_features_df = ioio_tki_features_df.query('BRAF_status == "positive"')

In [44]:
ioio_tki_features_df.shape

(1339, 174)

## Export dataframe

In [45]:
ioio_tki_features_df.to_csv('../outputs/ioio_tki_features_df.csv', index = False)

In [46]:
# Save dtypes
ioio_tki_features_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/ioio_tki_features_dtypes.csv')